# 05 · Features and labels: from 5-minute readings to a training table

**Who this is for:** you, after notebook 04.
**You should already know:** readings carry `device_id` only (02), outcomes arrive as events (03), devices are reused (03b).

**By the end you will be able to:**
- link each reading to the right patient using **device + time window**
- roll readings into **hourly buckets**, then into **features at a prediction time** (1 h / 3 h / 6 h windows, trends, missingness)
- compute **NEWS2**, the clinical baseline score
- build the **"escalated within 6 hours?" label**, and know when it's still unknown
- prove the table is **point-in-time correct**, and see how easily **leakage** makes a feature look magical

**Outline**
1. Setup: the 14-day baseline hospital
2. Step 1: link readings to patients
3. Step 2: hourly buckets
4. Step 3: features at a prediction time
5. Step 4: NEWS2
6. Step 5: labels
7. The finished training table
8. Leakage on purpose
9. Save the table
10. Exercise
11. Pitfalls and next steps

Code used: `databricks/src/heavden/ml/features.py`, `databricks/src/heavden/ml/news2.py` (tests in `tests/test_features.py`). In Databricks the gold pipeline must produce the same table.

## 1. Setup: the 14-day baseline hospital

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

from generator import activity, hospital, synthea, truth
from heavden.ml import features


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found")


ROOT = find_repo_root()
RUN = synthea.SyntheaRun()
csv_dir = synthea.run_synthea(RUN, ROOT / "data" / "synthea" / f"p{RUN.population}_s{RUN.seed}")
profiles = synthea.build_profiles(
    synthea.load_tables(csv_dir), pd.Timestamp(RUN.reference_date, tz="UTC")
)
inpatients = hospital.admit_inpatients(profiles)
START, DAYS = hospital.SIM_START, 14

In [ ]:
%%time
plan = activity.schedule_activity(profiles, inpatients, START, hours=24 * DAYS, seed=42)
ward = truth.simulate_ward(inpatients, START, hours=24 * DAYS, seed=42, activity=plan)
readings, records, outcomes = ward.readings, ward.activity, ward.truth.outcomes
nurse_obs = ward.nurse_observations
print(
    f"{len(readings):,} readings | {len(records.encounters):,} stays | {len(outcomes)} escalations"
)
print(f"{len(nurse_obs):,} nurse observations (ACVPU + oxygen)")

This is the **day-0 baseline** from Plan.md §6.4: 14 simulated days of normal hospital life. The model will be trained on it.

From here on we only use what the **data platform** would see:
- `readings`: device messages (Bronze vitals)
- `records`: encounters, device assignments, transfers, patient profiles (from Azure SQL)
- `outcomes`: escalation events
- `nurse_obs`: charted consciousness (ACVPU) and oxygen (from the nurses, `generator/care.py`)

Never `ward.truth.trajectories`, which is the hidden answer key.

## 2. Step 1: link readings to patients

A reading says *which device*, not *which patient*. Devices are reused, so the join must also check the **time**: a reading belongs to the assignment where `start_ts ≤ ts < end_ts` on that device.

In [ ]:
reused = records.device_assignments["device_id"].value_counts()
device = reused.index[0]
records.device_assignments.query("device_id == @device")[
    ["device_id", "encounter_id", "start_ts", "end_ts"]
]

In [ ]:
linked = features.link_readings(readings, records.device_assignments)
print(f"readings linked to a stay: {len(linked):,} of {len(readings):,}")

naive = readings.merge(records.device_assignments[["device_id", "encounter_id"]], on="device_id")
print(f"naive join on device_id only: {len(naive):,} rows. Every reading copied once per wearer!")

The naive join multiplies rows and mixes patients. `merge_asof` (in Spark, a range join) gets it right.

## 3. Step 2: hourly buckets

Readings are grouped into one-hour buckets `[b, b+1h)` per stay, with mean, median, min, max and count for each vital.

In [ ]:
buckets = features.hourly_buckets(linked)

# An escalated stay whose breathing rose most over its last 4 hours (picked from the data alone)
last = (
    buckets[buckets["encounter_id"].isin(outcomes["encounter_id"])].groupby("encounter_id").tail(5)
)
rise = last.groupby("encounter_id")["resp_rate_mean"].agg(lambda s: s.iloc[-1] - s.iloc[0])
example = rise.idxmax()
cols = ["n_readings", "heart_rate_mean", "resp_rate_mean", "spo2_min", "temp_c_mean", "sbp_mean"]
buckets[buckets["encounter_id"] == example].set_index("bucket")[cols].tail(8).round(1)

This stay ends in an escalation. Watch breathing (and usually heart rate) climb and SpO2 fall in the last few buckets: the signal the model must learn. Not every escalation looks this clear (sudden events, cardiac or septic patterns), which is why the model needs many features.

## 4. Step 3: features at a prediction time

The model makes a prediction **on the hour** for every patient on the ward. For prediction time **t**, features may only use buckets that **ended by t**:

```
buckets:   [t-6h ... t-5h) ... [t-3h, t-2h) [t-2h, t-1h) [t-1h, t)  │  t  │ [t, t+1h) ...
           ◄──────────── 6 h window ──────────────────────────────► │     │  FUTURE: never used
                                    ◄──── 3 h window ────────────►  │
                                                       ◄─ 1 h ──►   │
```

| Feature family | Example | Meaning |
|---|---|---|
| Last hour | `spo2_min_1h`, `heart_rate_median_1h` | What's happening right now |
| Longer windows | `resp_rate_mean_6h` | The recent baseline |
| Trend | `resp_rate_trend_3h` | Change in the hourly mean over 3 h. Deterioration shows up here. |
| Missingness | `missing_1h`, `missing_6h` | How many expected readings never arrived |
| Nurse charting | `new_confusion`, `on_oxygen`, `o2_flow_lpm`, `hours_since_obs` | The latest observation charted **before** t |
| Context | `age`, `copd`, `unit_type`, `hours_since_admission` | Who the patient is and where they are |

In [ ]:
%%time
table = features.build_patient_hours(readings, records, outcomes, nurse_observations=nurse_obs)
print(f"{len(table):,} patient-hours | {table.shape[1]} columns")

In [ ]:
show = [
    "spo2_min_1h",
    "spo2_mean_6h",
    "resp_rate_mean_1h",
    "resp_rate_trend_3h",
    "heart_rate_trend_3h",
    "missing_1h",
    "news2_total",
    "label",
]
table[table["encounter_id"] == example].set_index("prediction_ts")[show].tail(8).round(2)

## 5. Step 4: NEWS2

**NEWS2** is the early-warning score UK hospitals use. Seven parameters earn points and the total triggers clinical review. It's our **baseline**: the model must beat it.

| Points | 3 | 2 | 1 | 0 | 1 | 2 | 3 |
|---|---|---|---|---|---|---|---|
| Breathing (/min) | ≤8 | | 9–11 | 12–20 | | 21–24 | ≥25 |
| SpO2 scale 1 (%) | ≤91 | 92–93 | 94–95 | ≥96 | | | |
| Systolic BP | ≤90 | 91–100 | 101–110 | 111–219 | | | ≥220 |
| Heart rate | ≤40 | | 41–50 | 51–90 | 91–110 | 111–130 | ≥131 |
| Temperature (°C) | ≤35.0 | | 35.1–36.0 | 36.1–38.0 | 38.1–39.0 | ≥39.1 | |
| Supplemental oxygen | | Yes | | No | | | |
| Consciousness (ACVPU) | C, V, P or U | | | Alert | | | |

COPD patients use **SpO2 scale 2** (88–92% is fine for them, and high SpO2 *on oxygen* scores points, because too much oxygen is risky in COPD).
Vitals use the **median of the last hour**, so one artefact spike can't change the score. Consciousness and oxygen come from the **latest nurse observation before t**.

In [ ]:
closed = table[table["label"].notna()]
by_score = closed.groupby(closed["news2_total"].clip(upper=7))["label"].agg(["size", "mean"])
by_score.columns = ["patient-hours", "share escalating within 6 h"]
by_score.index = [f"{int(i)}" if i < 7 else "7+" for i in by_score.index]
by_score.round(3)

**Interpretation:** most patient-hours score 0 and very rarely escalate. The risk climbs steeply with the score, so NEWS2 is a **strong** baseline. Beating it will take real signal (trends, history, context).

## 6. Step 5: labels

For prediction time **t**: `label = 1` if an escalation happens in **(t, t + 6 h]**.
The answer is only known at `label_known_at = t + 6 h`. For the last hours of the data that moment hasn't arrived, so the label is **missing (NaN), not 0**.

In [ ]:
counts = (
    table["label"]
    .map({1.0: "1 (escalated within 6 h)", 0.0: "0 (did not)"})
    .fillna("unknown (window still open)")
)
counts.value_counts().to_frame("patient-hours")

In [ ]:
print(f"positive rate among known labels: {table['label'].mean():.2%}")
last = table["prediction_ts"].max()
print(
    "unknown labels are all in the last 6 hours:",
    bool((table.loc[table["label"].isna(), "prediction_ts"] > last - pd.Timedelta(hours=6)).all()),
)

## 7. The finished training table

`features.feature_columns(table)` lists what a model may use: everything except IDs, time and the label itself.

In [ ]:
cols = features.feature_columns(table)
print(len(cols), "feature columns")
missing = table[cols].isna().mean()
missing[missing > 0].sort_values(ascending=False).head(8).round(3).to_frame("share missing")

Some features are missing when a device was offline (dead battery) or the patient was just admitted (no 3 h history yet). LightGBM handles missing values natively, and `missing_1h` / `missing_6h` tell the model *that* data is missing, which is a signal in itself.

### Proving point-in-time correctness

The tests in `tests/test_features.py` check this with a trick: readings from 06:00 onwards get an absurd heart rate of 200, and the feature row for 06:00 must not see it. Let's run the same check on the real table:

In [ ]:
row = table.sample(1, random_state=1).iloc[0]
enc, t = row["encounter_id"], row["prediction_ts"]
window = linked[
    (linked["encounter_id"] == enc)
    & (linked["ts"] >= t - pd.Timedelta(hours=1))
    & (linked["ts"] < t)
]
print(f"{enc} at {t}: max heart rate in readings from the last hour = {window['heart_rate'].max()}")
print(f"feature heart_rate_max_1h = {row['heart_rate_max_1h']}")

## 8. Leakage on purpose

**Leakage** means a feature quietly contains information from the future. The model then looks brilliant in testing and fails in real use.
Let's make three leaky features and compare their predictive power with honest ones, measured by **AUROC** (0.5 = coin flip, 1.0 = perfect ranking).

In [ ]:
def auroc(score, label):
    """Chance a random positive scores higher than a random negative (ties count half)."""
    ok = ~(np.isnan(score) | np.isnan(label))
    s, y = score[ok], label[ok]
    u = stats.mannwhitneyu(s[y == 1], s[y == 0]).statistic
    return u / ((y == 1).sum() * (y == 0).sum())


known = table[table["label"].notna()].copy()
y = known["label"].to_numpy()

# Leaks 1 and 2: readings from the NEXT hour [t, t+1h), taken straight from the buckets
future = buckets.rename(columns={"bucket": "prediction_ts"}).assign(
    prediction_ts=lambda d: features._ns(d["prediction_ts"])
)[["encounter_id", "prediction_ts", "spo2_min", "resp_rate_max"]]
peek = known[["encounter_id", "prediction_ts"]].merge(future, how="left")
# Leak 3: "the stay ends within 6 hours", from the final discharge time
# (escalated stays end at the escalation, so this is the label in disguise)
end = known["encounter_id"].map(records.encounters.set_index("encounter_id")["discharge_ts"])
ends_soon = ((end - known["prediction_ts"]) <= pd.Timedelta(hours=6)).astype(float)

candidates = {
    "honest: NEWS2 total": known["news2_total"],
    "honest: -SpO2 min last hour": -known["spo2_min_1h"],
    "honest: breathing trend 3 h": known["resp_rate_trend_3h"],
    "LEAK: -SpO2 min NEXT hour": -peek["spo2_min"],
    "LEAK: breathing max NEXT hour": peek["resp_rate_max"],
    "LEAK: stay ends within 6 h": ends_soon,
}
pd.Series(
    {name: auroc(s.to_numpy(dtype=float), y) for name, s in candidates.items()}, name="AUROC"
).round(3)

**Interpretation:**
- "stay ends within 6 h" looks almost perfect, because escalated stays **end at the escalation**. In live use you can't know when a stay will end. It's the label in disguise, and the classic leak from using *final* records.
- the next-hour peeks are only somewhat better than the honest last-hour features, because deterioration builds gradually: the last hour already carries much of the signal. They're still invalid, and in a model they'd quietly inflate every metric.
- honest features are good but imperfect, which is what real early warning looks like.

Rule of thumb: **if a single feature looks too good to be true, check when its value becomes known.**

## 9. Save the table

This is the local equivalent of the gold tables `gold.patient_hour_features` and `gold.training_set`. Notebook 06 trains on it.

In [ ]:
out = ROOT / "data" / "gold_local"
out.mkdir(parents=True, exist_ok=True)
path = out / "patient_hours_baseline.parquet"
table.to_parquet(path, index=False)
print(f"saved {len(table):,} rows to {path.relative_to(ROOT)} (git-ignored)")

## 10. Exercise: which single vital is most predictive?

Using `known` and the `auroc` helper:
1. compute the AUROC of each **NEWS2 component** (`news2_resp_rate`, `news2_spo2`, `news2_oxygen`, `news2_sbp`, `news2_heart_rate`, `news2_consciousness`, `news2_temp`) and of `news2_total`
2. compute the AUROC of the **trend** features (`<vital>_trend_3h`, using the sign that makes "worse" higher, e.g. `-spo2_trend_3h`)
3. which single signal is strongest? Does any trend beat the NEWS2 total?

In [ ]:
# Try it yourself
component_auroc = None
trend_auroc = None

In [ ]:
# Answer (run after trying)
news2_cols = [
    "news2_resp_rate",
    "news2_spo2",
    "news2_oxygen",
    "news2_sbp",
    "news2_heart_rate",
    "news2_consciousness",
    "news2_temp",
    "news2_total",
]
component_auroc = pd.Series(
    {c: auroc(known[c].to_numpy(dtype=float), y) for c in news2_cols}
).round(3)
worse_is_higher = {"heart_rate": 1, "resp_rate": 1, "spo2": -1, "temp_c": 1, "sbp": -1}
trend_auroc = pd.Series(
    {
        f"{v}_trend_3h": auroc(sign * known[f"{v}_trend_3h"].to_numpy(dtype=float), y)
        for v, sign in worse_is_higher.items()
    }
).round(3)
pd.concat({"NEWS2 components": component_auroc, "trends": trend_auroc})

NEWS2 combines several weak signals into a strong one, and trends add information NEWS2 ignores. That's the opportunity the ML model will exploit in notebook 06.

## 11. Pitfalls and next steps

**Pitfall: joining by device only.** Devices are reused; always join on device **and** time window.

**Pitfall: off-by-one windows.** The bucket `[t, t+1h)` is the *future* at time `t`. Our features at `t` come from buckets ending at or before `t`, and a test checks it.

**Pitfall: treating unknown labels as 0.** The last 6 hours have open windows. Labelling them 0 tells the model "recent patients never escalate".

**Pitfall: features from final records.** Discharge time, length of stay and "worst value of the stay" are only known after the fact. In Databricks, features must come from data **as of t** (point-in-time joins in the feature store).

**Optional extension:** add a `spo2_drop_from_admission` feature (latest SpO2 minus the patient's first-hour SpO2) and check its AUROC. It's honest if the first hour is before t.

**Next notebook:** `06_news2_and_first_model`. A time-based split, the NEWS2 baseline, a LightGBM model with class weighting and calibration, and the first head-to-head.